In [ ]:
import numpy as np
import json
import torch
from pathlib import Path

from src.models.TransformerBottleneck_model import TransformerBottleneck_model
from src.models.model_utilizer import load_net
from src.dataloaders.ZerosPolesDataset import ConversionTransforms

In [ ]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version used by PyTorch: {torch.version.cuda}")
else:
    print("CUDA is not available. PyTorch is using CPU.")

## Настройки

In [ ]:
threshold = 0.5

Пути к каталогам.

In [ ]:
config_dir = Path("./src/config/")
log_name = 'TransformerBottleneck-model_halfwindow_8'

config_path = config_dir / "config.json"
assert config_path.exists(), f"Config not found: {config_path}"
with open(config_path, "r") as f:
    general_config = json.load(f)

checkpoints_dir = Path(general_config['checkpoints_dir'])
logs_dir = Path(general_config['logs_dir'])

model_log_path = logs_dir / f"{log_name}.json"
assert model_log_path.exists(), f"Config not found: {model_log_path}"
with open(model_log_path, "r") as f:
    model_log = json.load(f)

train_log = model_log["train_log"]
best_epoch = model_log["summary"]["best_epoch"]
dataset_path = Path(general_config['data_dir']) / model_log["metadata"]["dataset"]["dataset_name"]

In [ ]:
device = torch.device(general_config["device"].lower() if torch.cuda.is_available() else 'cpu')

In [ ]:
np.loadtxt(sample_path, delimiter=',', skiprows=1).T

transform = ConversionTransforms(
        num_iter=2,
        return_input=False
        )

inputs = transform(data)

In [ ]:
model = TransformerBottleneck_model(
    in_channels = 4,
    out_channels = 4,
    features = model_log["metadata"]["model"]["feature_list"]
    )

model = model.to(device)
model, epoch_init, _, _ = load_net(
            net = model,
            checkpoints_file = checkpoints_dir / "best_TransformerBottleneck-model_halfwindow_8.pth",
            device = device
            )

model.eval()

with torch.no_grad():
    logits = model(inputs.to(device))
    predictions = (torch.sigmoid(logits) > threshold).float()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


# Sample data
x = np.linspace(0, 10, 200)
y = np.sin(x) + np.random.normal(0, 0.1, size=x.shape)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y, linewidth=1.5, label='Data')

# Highlight vertical region from x=3 to x=5
ax.axvspan(3.5, 4.0, color='orange', alpha=0.25, label='Region of Interest')
ax.axvspan(3.75, 4.25, color='red', alpha=0.25, label='Region of Interest')

ax.set_xlabel('X-axis')
ax.set_ylabel('Y-axis')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()